# Mapa de regras do Radar Financeiro

Este notebook consolida as principais regras do Radar em uma tabela simples, com uma linha por tema, indicador e situacao.

A leitura segue a estrutura esperada:

| Tema | Indicador | Situacao | Clientes | % da Base |
|---|---|---|---:|---:|

Blocos mapeados:

- Base analisada e elegibilidade do Radar.
- Perfis usados como premissa: renda, macroperfil, microperfil e perfil financeiro.
- Movimentacao minima: transacoes, valores de entrada/saida e movimentacao Agro.
- Resultado orcamentario.
- Pontuacao por concentracao.
- Pontuacao por orcamento.
- Pontuacao por perfil.
- Pontuacao final por tema e status da decisao.

In [ ]:
%%spark

from pyspark.sql import functions as F

# Tabela final do Radar. Pode ser sobrescrita antes da execucao com a variavel tabela_spark.
tabela_origem = globals().get("tabela_spark", "sbx_t2i2016.v1_ana_edu_fin_cli")

df = spark.table(tabela_origem).dropDuplicates(["CD_CLI"])

tem_saida_total = F.coalesce(F.col("VL_SAI_TOTAL"), F.lit(0)) > 0

pontuacoes_finais = [
    "NR_PONT_CATEG",
    "NR_PONT_ORC",
    "NR_PONT_CONS",
    "NR_PONT_RES",
    "NR_PONT_CRED",
]

df = (
    df
    .withColumn(
        "NR_MAIOR_PONTUACAO",
        F.greatest(*[F.coalesce(F.col(c), F.lit(0)) for c in pontuacoes_finais])
    )
    .withColumn(
        "QT_TEMAS_MAIOR_PONTUACAO",
        sum(
            F.when(
                (F.col("NR_MAIOR_PONTUACAO") > 0)
                & (F.coalesce(F.col(c), F.lit(0)) == F.col("NR_MAIOR_PONTUACAO")),
                F.lit(1)
            ).otherwise(F.lit(0))
            for c in pontuacoes_finais
        )
    )
)

def regra(tema, indicador, situacao, ordem_tema, ordem_indicador):
    return F.struct(
        F.lit(ordem_tema).alias("ORDEM_TEMA"),
        F.lit(ordem_indicador).alias("ORDEM_INDICADOR"),
        F.lit(tema).alias("TX_TEMA"),
        F.lit(indicador).alias("TX_INDICADOR"),
        situacao.cast("string").alias("TX_SITUACAO")
    )

def classificar_perfil(coluna):
    valor = F.upper(F.trim(F.coalesce(F.col(coluna).cast("string"), F.lit("A CLASSIFICAR"))))
    return F.when(valor == "A CLASSIFICAR", F.lit("A CLASSIFICAR")).otherwise(F.lit("Com perfil"))

def maior_que_zero(coluna):
    return F.when(F.coalesce(F.col(coluna), F.lit(0)) > 0, F.lit("Maior que zero")).otherwise(F.lit("Igual a zero"))

def flag_sn(coluna, texto_sim, texto_nao):
    return F.when(
        F.upper(F.trim(F.coalesce(F.col(coluna).cast("string"), F.lit("N")))) == "S",
        F.lit(texto_sim)
    ).otherwise(F.lit(texto_nao))

def texto_coluna(coluna):
    return F.coalesce(F.col(coluna).cast("string"), F.lit("Nao informado"))

def pontos(coluna):
    valor = F.coalesce(F.col(coluna).cast("int"), F.lit(0))
    return F.concat(
        valor.cast("string"),
        F.when(valor == 1, F.lit(" ponto")).otherwise(F.lit(" pontos"))
    )

def pontuacao_tema(nome_tema, coluna):
    return F.concat(F.lit(nome_tema + " - "), pontos(coluna))

def resultado_orcamentario():
    return (
        F.when(F.col("CD_RES_ORC") == 1, F.lit("Superavitario"))
         .when(F.col("CD_RES_ORC") == 2, F.lit("Deficitario"))
         .when(F.col("CD_RES_ORC") == 0, F.lit("Neutro"))
         .otherwise(F.lit("Nao classificado"))
    )

def decisao_final():
    return (
        F.when(F.col("NR_MAIOR_PONTUACAO") == 0, F.lit("Sem prioridade"))
         .when(F.col("QT_TEMAS_MAIOR_PONTUACAO") > 1, F.lit("Empate"))
         .otherwise(F.lit("Tema unico"))
    )

def tema_no_topo(coluna_pontuacao):
    return F.when(
        (F.col("NR_MAIOR_PONTUACAO") > 0)
        & (F.coalesce(F.col(coluna_pontuacao), F.lit(0)) == F.col("NR_MAIOR_PONTUACAO")),
        F.lit("Sim")
    ).otherwise(F.lit("Nao"))

def conc_generica():
    return (
        F.when(~tem_saida_total, F.lit("Sem saida total - 0 ponto"))
         .when(F.col("NR_PONT_CONC_GEN") == 99, F.lit("Acima da referencia de genericos - 99 pontos"))
         .otherwise(F.lit("Dentro da referencia de genericos - 0 ponto"))
    )

def conc_essenciais():
    return (
        F.when(~tem_saida_total, F.lit("Sem saida total - 0 ponto"))
         .when(F.col("NR_PONT_CONC_ESS") == 0, F.lit("Abaixo de 50% das saidas - 0 ponto"))
         .when(F.col("NR_PONT_CONC_ESS") == 1, F.lit("De 50% a menos de 75% das saidas - 1 ponto"))
         .when(F.col("NR_PONT_CONC_ESS") == 2, F.lit("75% ou mais das saidas - 2 pontos"))
         .otherwise(F.lit("Nao classificado"))
    )

def conc_flexiveis():
    return (
        F.when(~tem_saida_total, F.lit("Sem saida total - 0 ponto"))
         .when(F.col("NR_PONT_CONC_FLEX") == 0, F.lit("Abaixo de 30% das saidas - 0 ponto"))
         .when(F.col("NR_PONT_CONC_FLEX") == 1, F.lit("De 30% a menos de 45% das saidas - 1 ponto"))
         .when(F.col("NR_PONT_CONC_FLEX") == 2, F.lit("45% ou mais das saidas - 2 pontos"))
         .otherwise(F.lit("Nao classificado"))
    )

def conc_reserva():
    return (
        F.when(~tem_saida_total, F.lit("Sem saida total - 0 ponto"))
         .when(F.col("NR_PONT_CONC_RES") == 0, F.lit("30% ou mais das saidas - 0 ponto"))
         .when(F.col("NR_PONT_CONC_RES") == 1, F.lit("De 20% a menos de 30% das saidas - 1 ponto"))
         .when(F.col("NR_PONT_CONC_RES") == 2, F.lit("Abaixo de 20% das saidas - 2 pontos"))
         .otherwise(F.lit("Nao classificado"))
    )

def conc_credito():
    return (
        F.when(~tem_saida_total, F.lit("Sem saida total - 0 ponto"))
         .when(F.col("NR_PONT_CONC_CRED") == 0, F.lit("Abaixo de 30% das saidas - 0 ponto"))
         .when(F.col("NR_PONT_CONC_CRED") == 1, F.lit("De 30% a menos de 45% das saidas - 1 ponto"))
         .when(F.col("NR_PONT_CONC_CRED") == 2, F.lit("45% ou mais das saidas - 2 pontos"))
         .otherwise(F.lit("Nao classificado"))
    )


In [ ]:
%%spark

linhas_regras = [
    regra("Base analisada", "Clientes unicos", F.lit("Total"), 10, 10),

    regra("Perfil de renda", "NM_PRFL", classificar_perfil("NM_PRFL"), 20, 10),
    regra("Macroperfil", "NM_MAC_PRFL_CLI", classificar_perfil("NM_MAC_PRFL_CLI"), 30, 10),
    regra("Microperfil", "NM_MIC_PRFL_CLI", classificar_perfil("NM_MIC_PRFL_CLI"), 40, 10),
    regra("Perfil financeiro", "NM_PRFL_FIN", classificar_perfil("NM_PRFL_FIN"), 50, 10),

    regra("Movimentacao Agro", "FL_TEM_MOV_AGRO", flag_sn("FL_TEM_MOV_AGRO", "Possui", "Nao possui"), 60, 10),
    regra("Transacoes totais", "QT_TRANS_TOTAL", maior_que_zero("QT_TRANS_TOTAL"), 70, 10),
    regra("Transacoes de entrada", "QT_TRANS_ENT", maior_que_zero("QT_TRANS_ENT"), 80, 10),
    regra("Transacoes de saida", "QT_TRANS_SAI", maior_que_zero("QT_TRANS_SAI"), 90, 10),
    regra("Valores de entrada", "VL_TRANS_ENT", maior_que_zero("VL_TRANS_ENT"), 100, 10),
    regra("Valores de saida", "VL_TRANS_SAI", maior_que_zero("VL_TRANS_SAI"), 110, 10),

    regra("Resultado do Radar", "FL_PARTICIPA_RADAR", flag_sn("FL_PARTICIPA_RADAR", "Participa", "Nao participa"), 120, 10),

    regra("Resultado orcamentario", "CD_RES_ORC", resultado_orcamentario(), 200, 10),
    regra("Resultado orcamentario", "TX_STS_RES", texto_coluna("TX_STS_RES"), 200, 20),
    regra("Resultado orcamentario", "TX_STS_FINAL", texto_coluna("TX_STS_FINAL"), 200, 30),

    regra("Concentracao dos gastos", "NR_PONT_CONC_GEN", conc_generica(), 300, 10),
    regra("Concentracao dos gastos", "NR_PONT_CONC_ESS", conc_essenciais(), 300, 20),
    regra("Concentracao dos gastos", "NR_PONT_CONC_FLEX", conc_flexiveis(), 300, 30),
    regra("Concentracao dos gastos", "NR_PONT_CONC_RES", conc_reserva(), 300, 40),
    regra("Concentracao dos gastos", "NR_PONT_CONC_CRED", conc_credito(), 300, 50),

    regra("Pontuacao por orcamento", "NR_PONT_ORC_GEN", pontuacao_tema("Categorizacao", "NR_PONT_ORC_GEN"), 400, 10),
    regra("Pontuacao por orcamento", "NR_PONT_ORC_ESS", pontuacao_tema("Gestao do Orcamento", "NR_PONT_ORC_ESS"), 400, 20),
    regra("Pontuacao por orcamento", "NR_PONT_ORC_FLEX", pontuacao_tema("Consumo Planejado", "NR_PONT_ORC_FLEX"), 400, 30),
    regra("Pontuacao por orcamento", "NR_PONT_ORC_RES", pontuacao_tema("Formacao de Reserva", "NR_PONT_ORC_RES"), 400, 40),
    regra("Pontuacao por orcamento", "NR_PONT_ORC_CRED", pontuacao_tema("Uso Consciente do Credito", "NR_PONT_ORC_CRED"), 400, 50),

    regra("Pontuacao por perfil", "NR_PONT_PRFL_GEN", pontuacao_tema("Categorizacao", "NR_PONT_PRFL_GEN"), 500, 10),
    regra("Pontuacao por perfil", "NR_PONT_PRFL_ESS", pontuacao_tema("Gestao do Orcamento", "NR_PONT_PRFL_ESS"), 500, 20),
    regra("Pontuacao por perfil", "NR_PONT_PRFL_FLEX", pontuacao_tema("Consumo Planejado", "NR_PONT_PRFL_FLEX"), 500, 30),
    regra("Pontuacao por perfil", "NR_PONT_PRFL_RES", pontuacao_tema("Formacao de Reserva", "NR_PONT_PRFL_RES"), 500, 40),
    regra("Pontuacao por perfil", "NR_PONT_PRFL_CRED", pontuacao_tema("Uso Consciente do Credito", "NR_PONT_PRFL_CRED"), 500, 50),

    regra("Pontuacao final por tema", "NR_PONT_CATEG", pontuacao_tema("Categorizacao de gastos", "NR_PONT_CATEG"), 600, 10),
    regra("Pontuacao final por tema", "NR_PONT_ORC", pontuacao_tema("Gestao do Orcamento", "NR_PONT_ORC"), 600, 20),
    regra("Pontuacao final por tema", "NR_PONT_CONS", pontuacao_tema("Consumo Planejado", "NR_PONT_CONS"), 600, 30),
    regra("Pontuacao final por tema", "NR_PONT_RES", pontuacao_tema("Formacao de Reserva", "NR_PONT_RES"), 600, 40),
    regra("Pontuacao final por tema", "NR_PONT_CRED", pontuacao_tema("Uso Consciente do Credito", "NR_PONT_CRED"), 600, 50),

    regra("Decisao da regra", "Status da decisao", decisao_final(), 700, 10),
    regra("Tema no topo", "Categorizacao de gastos", tema_no_topo("NR_PONT_CATEG"), 710, 10),
    regra("Tema no topo", "Gestao do Orcamento", tema_no_topo("NR_PONT_ORC"), 710, 20),
    regra("Tema no topo", "Consumo Planejado", tema_no_topo("NR_PONT_CONS"), 710, 30),
    regra("Tema no topo", "Formacao de Reserva", tema_no_topo("NR_PONT_RES"), 710, 40),
    regra("Tema no topo", "Uso Consciente do Credito", tema_no_topo("NR_PONT_CRED"), 710, 50),
]

df_regras_cliente = (
    df
    .select(
        "CD_CLI",
        F.explode(F.array(*linhas_regras)).alias("REGRA")
    )
    .select(
        "CD_CLI",
        F.col("REGRA.ORDEM_TEMA").alias("ORDEM_TEMA"),
        F.col("REGRA.ORDEM_INDICADOR").alias("ORDEM_INDICADOR"),
        F.col("REGRA.TX_TEMA").alias("TX_TEMA"),
        F.col("REGRA.TX_INDICADOR").alias("TX_INDICADOR"),
        F.col("REGRA.TX_SITUACAO").alias("TX_SITUACAO")
    )
)

qt_base = df.select(F.countDistinct("CD_CLI").alias("QT_BASE")).first()["QT_BASE"]

df_mapa_regras_radar = (
    df_regras_cliente
    .groupBy(
        "ORDEM_TEMA",
        "ORDEM_INDICADOR",
        "TX_TEMA",
        "TX_INDICADOR",
        "TX_SITUACAO"
    )
    .agg(F.countDistinct("CD_CLI").alias("QT_CLIENTES"))
    .withColumn("PC_BASE", F.round(F.col("QT_CLIENTES") / F.lit(qt_base) * 100, 2))
    .orderBy("ORDEM_TEMA", "ORDEM_INDICADOR", "TX_SITUACAO")
)

df_mapa_regras_radar.createOrReplaceTempView("vw_mapa_regras_radar")


In [ ]:
%%spark

df_mapa_regras_radar_exibicao = (
    df_mapa_regras_radar
    .select(
        F.col("TX_TEMA").alias("Tema"),
        F.col("TX_INDICADOR").alias("Indicador"),
        F.col("TX_SITUACAO").alias("Situacao"),
        F.translate(F.format_number(F.col("QT_CLIENTES"), 0), ",.", ".,").alias("Clientes"),
        F.concat(
            F.translate(F.format_number(F.col("PC_BASE"), 2), ",.", ".,"),
            F.lit("%")
        ).alias("% da Base")
    )
)

display(df_mapa_regras_radar_exibicao)


In [ ]:
%%spark

# Consulta direta, caso queira consumir a versao tecnica em SQL.
spark.sql("""
    SELECT
        TX_TEMA AS Tema,
        TX_INDICADOR AS Indicador,
        TX_SITUACAO AS Situacao,
        QT_CLIENTES AS Clientes,
        PC_BASE AS PC_Base
    FROM vw_mapa_regras_radar
    ORDER BY
        ORDEM_TEMA,
        ORDEM_INDICADOR,
        TX_SITUACAO
""").show(200, truncate=False)
